In [387]:
from google.cloud import bigquery
from google.cloud.exceptions import NotFound
import pickle

# Loading Sample Data and CPC/IPC Codes

In [20]:
data = pickle.load(open('data/multilingual_sample_1000en500nen_perCat_10encoders.pckl','rb'))

In [21]:
client  = bigquery.Client()

In [60]:
patent_ids = f"'{data[0]['patent_id']}'"
for item in data:
    patent_ids += f""",'{item['patent_id']}'"""

query_text = f"""SELECT patent_id, cpcs, ipcs FROM unified_patents.classifications WHERE patent_id IN ({patent_ids})"""

In [88]:
query_job = client.query(query_text)
query_results = query_job.result()
cpc_ipc_codes = [row for row in query_results]

In [91]:
codes = [{'patent_id':item[0], 'cpcs':item[1], 'ipcs':item[2]} for item in cpc_ipc_codes]

In [101]:
for item in data:
    item['cpcs'] = []
    item['ipcs'] = []

In [102]:
for item in data:
    for test in codes:
        if item['patent_id'] == test['patent_id']:
            item['cpcs'] = test['cpcs']
            item['ipcs'] = test['ipcs']

# Finding Patent Families

In [118]:
families = {}
for item in data:
    family_id = item['family_id']
    try:
        families[family_id].append(item)
    except KeyError:
        families[family_id] = [item]

In [130]:
family_ids = families.keys()

In [131]:
cpc_codes = []
ipc_codes = []
for family_id in family_ids:
    for item in families[family_id]:
        cpc_codes.extend(item['cpcs'])
        ipc_codes.extend(item['ipcs'])

In [185]:
cpc_code_list = f"'{cpc_codes[0]}'"
cpc_code_high_level = f"'{cpc_codes[0].split()[0]}'"
for code in cpc_codes:
    code_high_level = code.split()[0]
    code_reformat = code.replace(" ","").replace("/","-")
    cpc_code_list += f", '{code_reformat}'"
    cpc_code_high_level += f", '{code_high_level}'"

In [195]:
query_text = f"SELECT code, description FROM cpc_codes.all_codes WHERE code IN ({cpc_code_list})"
query_job = client.query(query_text)
query_results = query_job.result()

In [180]:
cpc_code_titles = [row for row in query_results]
cpc_titles = [{'cpc':item[0], 'title':item[1]} for item in cpc_code_titles]

In [191]:
query_text = f"SELECT code, title FROM cpc_codes.cpc_titles WHERE code IN ({cpc_code_high_level})"
query_job = client.query(query_text)
query_results = query_job.result()

In [192]:
cpc_code_high_level_titles = [row for row in query_results]
#cpc_high_level_titles = [{'cpc':item[0], 'title':item[1]} for item in cpc_code_high_level_titles]

In [193]:
cpc_high_level_titles = [{'cpc':item[0], 'title':item[1]} for item in cpc_code_high_level_titles]

In [194]:
cpc_high_level_titles

[{'cpc': 'G09F', 'title': 'SEALS'},
 {'cpc': 'A24B', 'title': 'SNUFF'},
 {'cpc': 'A01B',
  'title': 'PARTS, DETAILS, OR ACCESSORIES OF AGRICULTURAL MACHINES OR IMPLEMENTS, IN GENERAL making or covering furrows or holes for sowing, planting, or manuring A01C5/00; soil working for engineering purposes E01, E02, E21; measuring areas for agricultural purposes G01B'},
 {'cpc': 'A23K', 'title': 'FODDER'},
 {'cpc': 'A01D', 'title': 'MOWING'},
 {'cpc': 'H04J',
  'title': 'MULTIPLEX COMMUNICATION transmission in general H04B; peculiar to transmission of digital information H04L5/00; systems for the simultaneous or sequential transmission of more than one television signal H04N7/08; in exchanges H04Q11/00; stereophonic systems H04S'},
 {'cpc': 'C12Y', 'title': 'ENZYMES'},
 {'cpc': 'B67C', 'title': 'FUNNELS'},
 {'cpc': 'F16H', 'title': 'GEARING'},
 {'cpc': 'F16C', 'title': 'BEARINGS'},
 {'cpc': 'D04B', 'title': 'KNITTING'},
 {'cpc': 'B65D', 'title': 'PACKAGES'},
 {'cpc': 'F16J', 'title': 'SEALING

In [362]:
import xml.etree.ElementTree as ET
import os

In [363]:
def find_cpc_files(xml_dir):
    """
    Function which finds all the xml files in a directory and returns a list of them for use in parsing later.
    :param :xml_dir: String for the directory of interest
    :return: list of strings representing the files
    """
    print('Finding files')
    xml_files = os.listdir('../cpc_codes/'+xml_dir)
    xml_files.sort()
    xml_files = [fil for fil in xml_files if fil.endswith(".xml")]
    return(xml_files)


In [364]:
def parse_xml_file(xml_dir, xml_file):
    """
    Reads in an xml file and returns a list of dictionary items, with the cpc code and corresponding description.
    :param xml_dir: String type, directory where the xml files are
    :param xml_file: String type, name of the actual file
    :return: List of dictionaries, where each item in the list is a dictionary with the cpc code and corresponding description
    """
    tree = ET.parse(xml_dir + '/' + xml_file) #parse the xml file
    root = tree.getroot() #get the xml tree
    code_descriptions = []
    for item in root.findall('definition-item'): #look at each cpc item in the file
        for symbol in item.findall('classification-symbol'): #find the cpc name
            code = symbol.text
        description = ''
        for text in item.findall('definition-title'): #find the cpc description
            description += text.text
            for child in text: #for annoying reasons there are often links to other codes, in the form of nested children
                if child.text is not None:
                    description += child.text
                if child.tail is not None:
                    description += child.tail
        code_descriptions.append({'code':code, 'description':description})
    return code_descriptions

In [365]:
def get_cpc_descriptions(xml_directory):
    xml_files = find_cpc_files(xml_directory)
    code_descriptions = []
    print('Loading files')
    for fil in xml_files:
        code_descriptions.extend(parse_xml_file(xml_directory, fil))
    return code_descriptions

In [367]:
code_descriptions = get_cpc_descriptions('../cpc_codes/FullCPCDefinitionXML202401')

Finding files
Loading files


In [431]:
schema = [
    bigquery.SchemaField("code", "STRING", mode='REQUIRED', description="CPC code"),
    bigquery.SchemaField("description", 'STRING', mode='REQUIRED', description="Full description of CPC code")
]
table_id = 'gcp-cset-projects.kq57_sandbox.code_descriptions_4'

try:
    table = client.delete_table(table_id)
except NotFound:
    pass

table = bigquery.Table(table_id, schema=schema)
table = client.create_table(table)

table = client.insert_rows_json(table_id, code_descriptions)

In [430]:
ddl_query = "CREATE OR REPLACE TABLE kq57_sandbox.code_descriptions"
job = client.query(ddl_query)
job.result()

BadRequest: 400 No column definitions in CREATE TABLE at [1:1]

Location: US
Job ID: 81623b30-fd1a-4e3a-bce2-7aebf493c78c


In [415]:
client.delete_table(table_id)

In [422]:
table = bigquery.Table(table_id, schema=schema)
client.create_table(table)

Table(TableReference(DatasetReference('gcp-cset-projects', 'kq57_sandbox'), 'code_descriptions'))

In [419]:
table_id

'gcp-cset-projects.kq57_sandbox.code_descriptions_2'